# Out-of-Order Execution in Jupyter

The cells in a Jupyter notebook can be executed in *any* order, not just top to bottom. The cells below illustrate three common ways this goes wrong.

## Example 1: Stale values

Run the next three cells in order, top to bottom.

In [ ]:
price = 100

In [ ]:
tax = price * 0.10
print("tax:", tax)

In [ ]:
total = price + tax
print("total:", total)

Now go back and change `price = 100` to `price = 200` in the first cell, and re-run *only* that cell. Then re-run *only* the last cell.

The printed total is `210`, but for a price of `200` the correct total is `220`. The middle cell wasn't re-run, so `tax` is still based on the old price.

## Example 2: Non-idempotent cells

A cell is *idempotent* if running it twice gives the same result as running it once. Cells that mutate state are often not idempotent.

In [ ]:
count = 0

In [ ]:
count = count + 1
print(count)

Run the cell above three times in a row. You'll see `1`, then `2`, then `3` — even though the code never changes.

The same trap shows up with mutable containers:

In [ ]:
items = []

In [ ]:
items.append("hello")
print(items)

Run it once: `['hello']`. Run it again: `['hello', 'hello']`. The cell looks the same, but its effect depends on how many times it's been run.

## Example 3: Ghosts of deleted cells

Run this cell:

In [ ]:
secret = "leftover"

Now *delete* the cell above. The variable `secret` is still in the kernel's memory — the code that defined it is gone, but the value is not.

In [ ]:
print(secret)

This works right now, but anyone who restarts the kernel and runs the notebook top to bottom will get a `NameError`.

## The rule

To verify a notebook is correct, **Restart the Kernel and Run All Cells** from top to bottom. If the notebook still produces the right results after that, its execution order is consistent with its cell order. If it doesn't, the notebook has hidden state that depends on the order in which you happened to run things.